# Silver: limpieza y tipado con carga incremental
Lee las 7 tablas de Bronze, aplica las reglas de limpieza aprobadas y las guarda en `dunnhumby.silver` con `MERGE` (insertar lo nuevo y actualizar lo que cambió). Las filas rechazadas van a `transaction_data_cuarentena`.

**Carga incremental por día:** `transaction_data`, su cuarentena y `coupon_redempt` solo procesan los días que faltan, desde el último día cargado hasta el parámetro `dia_hasta`. El dataset de Kaggle es estático, así que este parámetro simula la llegada diaria de datos. Las tablas de referencia (campañas, cupones, hogares y productos) no tienen día: se comparan completas con `MERGE` por su clave.


In [0]:
# SECCIÓN 0 · Parámetros y funciones comunes (reglas 1 y 2, y carga incremental con MERGE)
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Parámetros del Workflow. dia_hasta = 711 procesa todo el dataset; un valor menor simula que aún no llegan los días siguientes
dbutils.widgets.text("dia_hasta", "711")
dbutils.widgets.text("dias_reproceso", "0")   # días ya cargados que se vuelven a procesar, por si llegan datos tardíos
DIA_HASTA = int(dbutils.widgets.get("dia_hasta"))
DIAS_REPROCESO = int(dbutils.widgets.get("dias_reproceso"))
assert DIA_HASTA >= 1 and DIAS_REPROCESO >= 0, "Parámetros inválidos"


def ultimo_dia_cargado():
    """Último día que ya está en Silver, entre transaction_data y su cuarentena (0 si aún no existen)."""
    ultimo = 0
    for t in ["dunnhumby.silver.transaction_data", "dunnhumby.silver.transaction_data_cuarentena"]:
        if spark.catalog.tableExists(t):
            ultimo = max(ultimo, spark.table(t).agg(F.max("day")).first()[0] or 0)
    return ultimo


# Ventana de días de esta corrida: del día siguiente al último cargado (menos el reproceso) hasta dia_hasta
DIA_DESDE = max(1, ultimo_dia_cargado() + 1 - DIAS_REPROCESO)
VENTANA = f"t.day BETWEEN {DIA_DESDE} AND {DIA_HASTA}"   # condición sobre la tabla destino para el MERGE
print(f"Ventana de esta corrida: días {DIA_DESDE} a {DIA_HASTA}" if DIA_DESDE <= DIA_HASTA else
      f"No hay días nuevos: Silver ya tiene cargado hasta el día {DIA_DESDE - 1}")


def construir_silver(nombre, tipos):
    """Lee la tabla de Bronze y aplica las reglas comunes: nombres en minúsculas, tipos y texto sin espacios."""
    df = spark.table(f"dunnhumby.bronze.{nombre}").drop("_archivo_origen", "_fecha_carga")
    df = df.toDF(*[c.lower() for c in df.columns])   # regla 1: Bronze mezcla minúsculas y mayúsculas
    for c in df.columns:
        if c in tipos:
            # try_cast devuelve NULL si el texto no es convertible, en lugar de lanzar error
            df = df.withColumn(c, F.expr(f"try_cast(`{c}` AS {tipos[c]})"))
        else:
            df = df.withColumn(c, F.trim(F.col(c)))  # regla 2: texto sin espacios sobrantes
    return df.withColumn("_fecha_proceso", F.current_timestamp())


def merge_delta(df, tabla, claves, borrar_si=None):
    """MERGE por la clave natural: inserta las filas nuevas y actualiza solo las que cambiaron.
    borrar_si es una condición sobre la tabla destino: las filas que la cumplen y ya no vienen en df se borran.
    Devuelve cuántas filas se insertaron, actualizaron y borraron."""
    if not spark.catalog.tableExists(tabla):
        # Primera corrida: la tabla aún no existe y se crea con este lote
        df.write.format("delta").saveAsTable(tabla)
        return {"insertadas": spark.table(tabla).count(), "actualizadas": 0, "borradas": 0}

    merge = (DeltaTable.forName(spark, tabla).alias("t")
             .merge(df.alias("s"), " AND ".join(f"t.`{c}` = s.`{c}`" for c in claves)))
    # Una fila existente solo se reescribe si cambió algún dato; así, re-ejecutar no modifica nada (idempotencia)
    datos = [c for c in df.columns if c not in claves and c != "_fecha_proceso"]
    if datos:
        merge = merge.whenMatchedUpdateAll(condition=" OR ".join(f"NOT (t.`{c}` <=> s.`{c}`)" for c in datos))
    merge = merge.whenNotMatchedInsertAll()
    if borrar_si:
        merge = merge.whenNotMatchedBySourceDelete(condition=borrar_si)
    merge.execute()

    m = DeltaTable.forName(spark, tabla).history(1).first()["operationMetrics"]
    return {"insertadas": int(m.get("numTargetRowsInserted", 0)),
            "actualizadas": int(m.get("numTargetRowsUpdated", 0)),
            "borradas": int(m.get("numTargetRowsDeleted", 0))}


def guardar_silver(df, nombre, tipos, claves, borrar_si="true"):
    """Verifica que la conversión de tipos no haya creado NULL y hace MERGE en silver.
    Por defecto (borrar_si="true") la tabla queda igual a la fuente: lo que ya no viene en Bronze se borra."""
    nulos = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in tipos]).first()
    problemas = {c: nulos[c] for c in tipos if nulos[c]}
    assert not problemas, f"Valores no convertibles en {nombre}: {problemas}"

    r = merge_delta(df, f"dunnhumby.silver.{nombre}", claves, borrar_si)
    n_silver = spark.table(f"dunnhumby.silver.{nombre}").count()
    print(f"{nombre}: {r['insertadas']:,} insertadas, {r['actualizadas']:,} actualizadas, "
          f"{r['borradas']:,} borradas -> silver {n_silver:,} filas")


## 1. Tablas simples
Solo necesitan tipos y limpieza de texto: `campaign_desc`, `campaign_table` y `coupon_redempt`. Las dos primeras se comparan completas; `coupon_redempt` tiene día y entra solo la ventana de días de la corrida.


In [0]:
# SECCIÓN 1 · campaign_desc, campaign_table, coupon_redempt · solo tipos, MERGE por clave natural
TABLAS_SIMPLES = {
    # tabla: (tipos, clave natural)
    "campaign_desc":  ({"campaign": "int", "start_day": "int", "end_day": "int"}, ["campaign"]),
    "campaign_table": ({"household_key": "int", "campaign": "int"}, ["household_key", "campaign"]),
}
for nombre, (tipos, claves) in TABLAS_SIMPLES.items():
    guardar_silver(construir_silver(nombre, tipos), nombre, tipos, claves)

# coupon_redempt tiene día: entra solo la ventana de esta corrida, igual que transaction_data
tipos_red = {"household_key": "int", "day": "int", "coupon_upc": "bigint", "campaign": "int"}
df = construir_silver("coupon_redempt", tipos_red).filter(F.col("day").between(DIA_DESDE, DIA_HASTA))
guardar_silver(df, "coupon_redempt", tipos_red, ["household_key", "day", "coupon_upc", "campaign"], borrar_si=VENTANA)


In [0]:
# SECCIÓN 1 · Verificación de tipos de las tablas simples
# Esperado: columnas en minúsculas, identificadores integer/long y _fecha_proceso como timestamp
for nombre in ["campaign_desc", "campaign_table", "coupon_redempt"]:
    print(nombre)
    spark.table(f"dunnhumby.silver.{nombre}").printSchema()

## 2. coupon y hh_demographic
Tablas con reglas propias: duplicados en `coupon` y valores de relleno en `hh_demographic`.

In [0]:
# SECCIÓN 2 · coupon · regla 12: eliminar duplicados exactos; MERGE por las tres columnas
tipos_coupon = {"coupon_upc": "bigint", "product_id": "bigint", "campaign": "int"}
df = construir_silver("coupon", tipos_coupon)

# Regla 12: se eliminan las filas idénticas. _fecha_proceso se excluye de la comparación
columnas = [c for c in df.columns if c != "_fecha_proceso"]
df = df.dropDuplicates(columnas)

# Sin duplicados, las tres columnas juntas identifican cada fila
guardar_silver(df, "coupon", tipos_coupon, ["coupon_upc", "product_id", "campaign"])

In [0]:
# SECCIÓN 2 · hh_demographic · reglas 10 y 11: desconocidos a NULL y orden de ingreso
tipos_hh = {"household_key": "int"}
df = construir_silver("hh_demographic", tipos_hh)

# Regla 10: los valores de relleno pasan a NULL. kid_category_desc no se toca: "None/Unknown" mezcla dos significados
for c in ["hh_comp_desc", "homeowner_desc"]:
    df = df.withColumn(c, F.when(F.col(c) == "Unknown", None).otherwise(F.col(c)))
df = df.withColumn("marital_status_code",
                   F.when(F.col("marital_status_code") == "U", None).otherwise(F.col("marital_status_code")))

# Regla 11: como texto, "15-24K" ordena después de "125-149K"; income_orden da el orden real
ORDEN_INGRESO = ["Under 15K", "15-24K", "25-34K", "35-49K", "50-74K", "75-99K",
                 "100-124K", "125-149K", "150-174K", "175-199K", "200-249K", "250K+"]
mapa = F.create_map(*[x for i, r in enumerate(ORDEN_INGRESO, 1) for x in (F.lit(r), F.lit(i))])
df = df.withColumn("income_orden", mapa[F.col("income_desc")])

# Todos los rangos de Bronze deben estar en la lista; si no, income_orden quedaría NULL
assert df.filter("income_orden IS NULL").count() == 0, "Hay rangos de ingreso fuera de la lista"

guardar_silver(df, "hh_demographic", tipos_hh, ["household_key"])

In [0]:
# SECCIÓN 2 · Verificación de coupon y hh_demographic
# Esperado: coupon con 119,384 filas y 0 duplicadas; nulos de hh_comp 73, homeowner 233, marital 344; kid_none_unknown 558
c = spark.table("dunnhumby.silver.coupon").drop("_fecha_proceso")
print(f"coupon: {c.count():,} filas, {c.count() - c.distinct().count():,} duplicadas")

hh = spark.table("dunnhumby.silver.hh_demographic")
display(hh.select(
    F.sum(F.col("hh_comp_desc").isNull().cast("int")).alias("hh_comp_nulos"),
    F.sum(F.col("homeowner_desc").isNull().cast("int")).alias("homeowner_nulos"),
    F.sum(F.col("marital_status_code").isNull().cast("int")).alias("marital_nulos"),
    F.sum((F.col("kid_category_desc") == "None/Unknown").cast("int")).alias("kid_none_unknown"),
))
display(hh.groupBy("income_orden", "income_desc").count().orderBy("income_orden"))

## 3. product
Catálogo de productos: vacíos a `NULL`, productos sin clasificar y marca de no mercadería.

In [0]:
# SECCIÓN 3 · product · reglas 7, 8 y 9
tipos_product = {"product_id": "bigint", "manufacturer": "int"}
df = construir_silver("product", tipos_product)

# Regla 7: un tamaño vacío no es un valor, es la ausencia de dato
df = df.withColumn("curr_size_of_product",
                   F.when(F.col("curr_size_of_product") == "", None).otherwise(F.col("curr_size_of_product")))

# Regla 8: los 15 productos sin clasificar se conservan, con una etiqueta explícita
# La condición se guarda en una columna antes del ciclo: si se evaluara dentro, department ya estaría reemplazado
df = df.withColumn("_sin_clasificar", F.col("department").isNull() | (F.col("department") == ""))
for c in ["department", "commodity_desc", "sub_commodity_desc"]:
    df = df.withColumn(c, F.when(F.col("_sin_clasificar"), "SIN CLASIFICAR").otherwise(F.col(c)))
df = df.drop("_sin_clasificar")

# Regla 9: su cantidad no es comparable con la de la mercadería (la gasolina se mide en otra unidad)
NO_MERCADERIA = ["KIOSK-GAS", "MISC SALES TRAN", "MISC. TRANS."]
df = df.withColumn("es_no_mercaderia", F.col("department").isin(NO_MERCADERIA))

guardar_silver(df, "product", tipos_product, ["product_id"])

In [0]:
# SECCIÓN 3 · Verificación de product
# Esperado: size_nulos 30,607; 15 sin clasificar en las tres columnas; no_mercaderia 594 (KIOSK-GAS 16, MISC SALES TRAN 88, MISC. TRANS. 490)
pr = spark.table("dunnhumby.silver.product")

display(pr.select(
    F.sum(F.col("curr_size_of_product").isNull().cast("int")).alias("size_nulos"),
    F.sum((F.col("department") == "SIN CLASIFICAR").cast("int")).alias("dept_sin_clasificar"),
    F.sum((F.col("commodity_desc") == "SIN CLASIFICAR").cast("int")).alias("commodity_sin_clasificar"),
    F.sum((F.col("sub_commodity_desc") == "SIN CLASIFICAR").cast("int")).alias("sub_sin_clasificar"),
    F.sum(F.col("es_no_mercaderia").cast("int")).alias("no_mercaderia"),
))
display(pr.filter("es_no_mercaderia").groupBy("department").count().orderBy("department"))

## 4. transaction_data y cuarentena
Tabla central: importes redondeados, hora derivada y separación de las líneas sin unidades. Carga incremental: solo la ventana de días de la corrida, con `MERGE` por `basket_id` + `product_id` (la combinación es única en el dataset y cada canasta pertenece a un solo día).


In [0]:
# SECCIÓN 4 · transaction_data · reglas 3, 4, 5 y 6 (las líneas rechazadas van a cuarentena) · carga incremental por día
tipos_tx = {
    "household_key": "int", "basket_id": "bigint", "day": "int", "product_id": "bigint",
    "quantity": "int", "sales_value": "decimal(10,2)", "store_id": "int",
    "retail_disc": "decimal(10,2)", "trans_time": "int", "week_no": "int",
    "coupon_disc": "decimal(10,2)", "coupon_match_disc": "decimal(10,2)",
}
CLAVE_TX = ["basket_id", "product_id"]

# Regla 3: pasar a decimal(10,2) redondea los importes y elimina el ruido de punto flotante (2.22E-16)
# Solo entran los días de la ventana de esta corrida
df = construir_silver("transaction_data", tipos_tx).filter(F.col("day").between(DIA_DESDE, DIA_HASTA))

# Se comprueba que ningún valor haya quedado en NULL antes de separar las filas; si no, alguna fila no caería en ninguna tabla
nulos = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in tipos_tx]).first()
assert not any(nulos[c] for c in tipos_tx), "Valores no convertibles en transaction_data"

# Regla 6: trans_time viene como HHMM sin ceros a la izquierda (45 significa 00:45)
assert df.filter((F.col("trans_time") % 100) >= 60).count() == 0, "Hay minutos inválidos en trans_time"
df = df.withColumn("hora", F.floor(F.col("trans_time") / 100).cast("int"))

# Regla 4: una línea sin unidades no es una compra; va a cuarentena con su motivo
cantidad_cero = F.col("quantity") == 0
cuarentena = df.filter(cantidad_cero).withColumn(
    "motivo",
    F.when(F.col("sales_value") == 0, "cantidad y venta en cero")
     .otherwise("cantidad cero con venta positiva"),
)
principal = df.filter(~cantidad_cero)   # la regla 5 se cumple sola: las líneas con unidades y venta 0 se quedan aquí

# MERGE de los dos destinos; borrar_si=VENTANA: dentro de la ventana, lo que ya no viene en Bronze se elimina
r = merge_delta(cuarentena, "dunnhumby.silver.transaction_data_cuarentena", CLAVE_TX, VENTANA)
print(f"transaction_data_cuarentena: {r['insertadas']:,} insertadas, {r['actualizadas']:,} actualizadas, {r['borradas']:,} borradas")
guardar_silver(principal, "transaction_data", tipos_tx, CLAVE_TX, borrar_si=VENTANA)

# Cuadre: toda fila de Bronze hasta el último día cargado debe estar en una de las dos tablas
ultimo = ultimo_dia_cargado()
n_b = spark.table("dunnhumby.bronze.transaction_data").filter(F.expr(f"try_cast(DAY AS INT) <= {ultimo}")).count()
n_s = spark.table("dunnhumby.silver.transaction_data").count()
n_c = spark.table("dunnhumby.silver.transaction_data_cuarentena").count()
assert n_b == n_s + n_c, "Las filas no cuadran con Bronze"
print(f"Cuadre hasta el día {ultimo}: {n_b:,} = {n_s:,} (silver) + {n_c:,} (cuarentena)")


In [0]:
# SECCIÓN 4 · Verificación de transaction_data y su cuarentena
# Esperado con dia_hasta = 711: cuarentena de 14,466 filas en total (14,428 + 38); quantity_min 1; hora de 0 a 23
display(
    spark.table("dunnhumby.silver.transaction_data_cuarentena")
    .groupBy("motivo")
    .agg(F.count("*").alias("filas"), F.sum("sales_value").alias("venta_total"))
)

tx = spark.table("dunnhumby.silver.transaction_data")
display(tx.select(
    F.min("quantity").alias("quantity_min"),
    F.sum((F.col("sales_value") == 0).cast("int")).alias("venta_cero"),
    F.sum((F.col("retail_disc") > 0).cast("int")).alias("retail_disc_positivo"),
    F.min("hora").alias("hora_min"),
    F.max("hora").alias("hora_max"),
))